In [1]:
import jax
import jax_sn
import jax.numpy as jnp
from jax_sn.domain import GeometricDomain, CrossSectionDataSingleGroup
from jax_sn.solution_domain import FEMMatrices, PrecomputedGeometricData
from jax_sn.operators.transport_operator import TransportOperatorVmapPallas, TransportOperatorBatchedPallas
from jax_sn.domain import TopologicalSortedAngle
import equinox as eqx
from jax_sn.tools.memory_analysis import _filter_jit_f_to_memory
%load_ext autoreload
%autoreload 2

jax.config.update("jax_compilation_cache_dir", "/tmp/jax_cache")
jax.config.update("jax_persistent_cache_min_entry_size_bytes", -1)
jax.config.update("jax_persistent_cache_min_compile_time_secs", 0)

jax.config.update("jax_enable_x64", True)

In [30]:
dim =    3
degree = 1
resolution = 30
tn_order = 3
n_groups = 5
l_order = 4

jax.config.update("jax_log_compiles", False)

mgop = jax_sn.tools.setup_test_mgop.setup_slab_mgop(dim, degree, resolution, tn_order, n_groups, l_order, True, single_angle_transport_operator_cls = TransportOperatorBatchedPallas, single_angle_transport_operator_cls_kwargs = {"batch_size": 100})

In [31]:
mgop.n_elements

162000

In [32]:
xout = jnp.ones_like(mgop.transport_operator.out_structure())

In [38]:
from warp_kernel_v2 import WarpSweep
import numpy as onp
# top0 is a single-angle operator (defined earlier in the notebook)
sigma_t_jax = mgop.solution_domain_data.domain.cross_sections_indexed.map_to_full_array(0).total



warp_v2 = WarpSweep.from_transport_operator(mgop.transport_operator, sigma_t_jax)
print("SweepData built OK")

SweepData built OK


In [39]:
from warp_kernel_v2 import compute_inflow_and_omega_n_Je
import warp as wp
from jax_sn.operators.transport_operator.transport_operator_imp import _compute_omega_dot_n_Je_inflow
b = jnp.ones_like(mgop.transport_operator.out_structure())

result_warp = warp_v2.compute_all_angles(sigma_t_jax, b)

In [43]:
%timeit jax_result = mgop.transport_operator.inv(b)

473 ms ± 7.03 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [44]:
b = jnp.ones_like(mgop.transport_operator.out_structure())
result_jax = mgop.transport_operator.inv(b)
onp.testing.assert_allclose(result_jax, result_warp.numpy())

In [45]:
from warp_kernel_v2 import _compute_level_bounds, LevelBasedParallelismSetup


lsetup = LevelBasedParallelismSetup.from_transport_operator(mgop.transport_operator)

In [46]:
xs_wp = wp.from_jax(sigma_t_jax)


In [48]:
result_3 = warp_v2.multi_angle_levels(xs_wp, b, lsetup)

In [50]:
%timeit -n 10 -r 3 result_3 = warp_v2.multi_angle_levels(xs_wp, b, lsetup)

59.2 ms ± 10.2 ms per loop (mean ± std. dev. of 3 runs, 10 loops each)


In [2]:
result_3

NameError: name 'result_3' is not defined

In [54]:
onp.testing.assert_allclose(result_3.numpy(), result_jax)

In [55]:
%timeit -n 10 -r 3 result = warp_v2.single_angle(0, xs_wp, x, level_bounds)

The slowest run took 7.50 times longer than the fastest. This could mean that an intermediate result is being cached.
10.9 ms ± 9.55 ms per loop (mean ± std. dev. of 3 runs, 10 loops each)
